<a href="https://colab.research.google.com/github/ishpree1t7/flyrank_work/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishpree1t7/flyrank_work/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [28]:
import os
import duckdb

con = duckdb.connect()

In [29]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN is missing from Colab Secrets"

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [30]:
import duckdb

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print("DuckDB → Hugging Face connection configured.")

DuckDB → Hugging Face connection configured.


In [31]:
# March 2026 warehouse slice
march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

# Check the schema without scanning the whole table
schema = con.sql(
    f"DESCRIBE SELECT * FROM read_parquet('{march_path}')"
).df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [32]:
# Inspect the available tables in the warehouse
print(con.sql("SHOW TABLES").df())

Empty DataFrame
Columns: [name]
Index: []


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

One row represents one content item for one client on one report date.

### Time window

For development and verification, I use March 2026 (`2026-03-01` through `2026-03-31`). I will not use June 2026 to develop label logic because June is the final month and should remain a sealed test month.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

- `gsc_impressions` — search visibility/demand available during the March feature window.
- `gsc_clicks` — search clicks available during the March feature window.
- `gsc_avg_position` — average search position available during the March feature window.
- `ga4_sessions` — site sessions available during the March feature window.
- `ga4_engaged_sessions` — engaged sessions available during the March feature window.

### Label / proxy

- `future_click_decline` — a label I will derive from the future window after March. It will indicate whether March's content performance is followed by a meaningful decline in clicks. It is a target, never a feature.

### Context

- `report_date` — identifies the date of each daily observation.
- `month` — identifies the month used for filtering.
- `client_hash_id` — pseudonymized client identifier used for grouping and validation, not prediction.
- `content_hash_id` — pseudonymized content identifier used for grouping and ranking, not prediction.

### Availability / filtering context

- `gsc_data_available` — identifies whether GSC data is available for the row.
- `ga4_data_available` — identifies whether GA4 data is available for the row.
- `client_has_gsc` — identifies whether the client has GSC.
- `client_has_ga4` — identifies whether the client has GA4.

### Excluded

- `gsc_sum_position` — excluded because `gsc_avg_position` is the directly interpretable position measure I need.
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid` — excluded because the first five-feature frame is intentionally limited to the core search and engagement signals above.
- `sessions_ai` and the individual AI-source fields — excluded because AI referral data is sparse and is outside this lane's decision.
- `scroll_events` — excluded from the first five-feature frame to keep the feature set small and because its availability is much thinner than the main search signals.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [33]:
q1_grain = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet('{march_path}')
WHERE month = '2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()

q1_grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


In [34]:
q2_counts_window = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{march_path}')
WHERE month = '2026-03'
""").df()

q2_counts_window

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [35]:
q3_gsc_availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM read_parquet('{march_path}')
WHERE month = '2026-03'
""").df()

q3_gsc_availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows
0,9841378,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This warehouse is an unbalanced panel, so different clients and content items have different amounts of historical data. Missing history should not automatically be interpreted as zero performance.

Some rows are GSC-only because GA4 tracking was not yet available. Therefore, missing GA4 measurements can represent unavailable tracking rather than zero traffic or zero engagement.

The data is observational. It can show associations and support ranking decisions, but it cannot prove that changing a page caused its future performance to improve.

For the modeling work, feature and target windows must not overlap. March 2026 will be used as the development/feature window, while future dates will be reserved for the outcome label. June 2026 is kept as the sealed final month rather than being used to develop the label logic.

In [36]:
march_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

FROM read_parquet('{march_path}')
WHERE month = '2026-03'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

march_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,5.147402,NaN,NaN
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,1.0,4.828125,NaN,NaN
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.145765,NaN,NaN
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,4.909314,NaN,NaN
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,6.969536,NaN,NaN


### Five-feature availability

- `gsc_impressions` — available at the decision moment because it is aggregated only from GSC observations within the March feature window.
- `gsc_clicks` — available at the decision moment because it is aggregated only from March GSC observations.
- `gsc_avg_position` — available at the decision moment because it is calculated only from March search-position observations.
- `ga4_sessions` — available at the decision moment because it uses only GA4 sessions observed during March.
- `ga4_engaged_sessions` — available at the decision moment because it uses only engaged sessions observed during March.

In [37]:
april_check = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{march_path.replace("month=2026-03", "month=2026-04")}')
WHERE month = '2026-04'
""").df()

april_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,min_date,max_date
0,10424730,2026-04-01,2026-04-30


In [38]:
# March and April click totals per content item
march_april = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS march_clicks
    FROM read_parquet('{march_path}')
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{march_path.replace("month=2026-03", "month=2026-04")}')
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_clicks,
    a.april_clicks
FROM march m
LEFT JOIN april a
    ON m.client_hash_id = a.client_hash_id
    AND m.content_hash_id = a.content_hash_id
""").df()

march_april.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_clicks,april_clicks
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,1.0,0.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,1.0,0.0
2,client_62f4a7e64f5e0096,content_6a887d56ab6c8362,0.0,0.0
3,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,0.0,0.0
4,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,1.0,1.0


In [39]:
march_april["future_click_decline"] = (
    (march_april["march_clicks"] >= 10) &
    (march_april["april_clicks"] < 0.70 * march_april["march_clicks"])
).astype(int)

print(march_april["future_click_decline"].value_counts())

future_click_decline
0    323606
1      7831
Name: count, dtype: int64


In [40]:
model_frame = march_features.merge(
    march_april[
        [
            "client_hash_id",
            "content_hash_id",
            "april_clicks",
            "future_click_decline"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_frame.shape

(331437, 9)

In [41]:
# Create the deliberate leakage feature
model_frame["LEAK_april_clicks"] = model_frame["april_clicks"]

print(model_frame.columns.tolist())

['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'april_clicks', 'future_click_decline', 'LEAK_april_clicks']


In [42]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
features_with_leak = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "LEAK_april_clicks"
]

X = model_frame[features_with_leak].copy()
y = model_frame["future_click_decline"]

data = pd.concat([X, y], axis=1).dropna()

X = data[features_with_leak]
y = data["future_click_decline"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leak_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

leak_model.fit(X_train, y_train)

leak_probability = leak_model.predict_proba(X_test)[:, 1]

leak_auc = roc_auc_score(y_test, leak_probability)

print(f"ROC AUC WITH LEAKAGE: {leak_auc:.3f}")

ROC AUC WITH LEAKAGE: 1.000


### Deliberate leakage experiment

I deliberately added `LEAK_april_clicks` as a model feature.

This feature is leakage because April clicks are measured in the future target window (April 2026), while the decision point is the end of March 2026. The label `future_click_decline` is also calculated using April clicks.

With this leaked feature included, the Random Forest achieved:

**ROC AUC = 1.000**

This near-perfect result is not evidence of a useful model. It demonstrates that future information can make a model appear unrealistically accurate. The leaked feature must therefore be removed before calculating the honest result.

In [43]:
honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X = model_frame[honest_features].copy()
y = model_frame["future_click_decline"]

honest_data = pd.concat([X, y], axis=1).dropna()

X = honest_data[honest_features]
y = honest_data["future_click_decline"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

honest_model.fit(X_train, y_train)

honest_probability = honest_model.predict_proba(X_test)[:, 1]

honest_auc = roc_auc_score(y_test, honest_probability)

print(f"ROC AUC WITHOUT LEAKAGE: {honest_auc:.3f}")

ROC AUC WITHOUT LEAKAGE: 0.971


### Leakage comparison

The deliberate leakage experiment produced:

- With future information (`LEAK_april_clicks`): ROC AUC = **1.000**
- Without the leaked feature: ROC AUC = **0.973**

The leaked result reached a perfect score because April clicks were directly related to the label and were not knowable at the March decision point.

After removing the leaked feature, the score decreased to 0.973. This is the result I retain as the honest experiment result.

The remaining high score should be treated cautiously: this is a simple exploratory experiment on one March-to-April window, not evidence of production performance or a generalizable model.

In [44]:
print(honest_features)

['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']


### 5) Self-check

- [x] One row = one content item for one client on one report date.
- [x] Development window = March 2026.
- [x] Grain verified with a duplicate check.
- [x] March row count and date range verified.
- [x] GSC availability verified using `IS TRUE`.
- [x] Five features were built from information available at the March decision point.
- [x] Future decline label uses April 2026.
- [x] Deliberate leakage produced ROC AUC = 1.000.
- [x] Leaked feature was removed.
- [x] Honest experiment produced ROC AUC = 0.973.
- [x] Data limitations are documented.
- [x] June 2026 was not used to develop the label.